# Efficient LLM Serving at Scale with Unified Caching

**Author**: Haichen Zhang  
**Knowledge level**: Intermediate

This notebook walks through two LMCache benchmark workflows on AMD Instinct GPUs using `Qwen/Qwen3.6-35B-A3B`:

- **Long Document QA**: repeated questions over long documents to measure time-to-first-token (TTFT) improvement when KV cache is reused.
- **Multi-round QA**: concurrent stateful chat sessions where conversation history grows across turns.

In this tutorial, you'll learn how to use LMCache to unlock AMD GPU acceleration and experience significant performance improvements in real-world enterprise use cases, including Long-Document QA and Multi-Round QA applications.

## What you will learn

In this tutorial, you will:

1. Set up a ROCm/vLLM environment for Qwen3.6-35B-A3B.
2. Build LMCache from source with HIP support for AMD GPUs.
3. Launch three serving configurations: no prefix cache, vLLM HBM prefix cache, and LMCache CPU-DRAM KV offload.
4. Validate the OpenAI-compatible endpoint and check whether LMCache hits are visible.
5. Run Long Document QA and Multi-round QA benchmarks.
6. Compare TTFT, latency, throughput, and cache behavior across configurations.

The benchmark goal is not only to show cache hits, but to identify when LMCache's L2 CPU-memory tier helps: long-context, repeated, or multi-turn workloads where the active KV working set pressures GPU HBM.

## Prerequisites

This tutorial was designed for an AMD GPU server(Radeon GPU or Instinct GPU platform) with ROCm support.

### Operating system

- Ubuntu 22.04 or 24.04.

### Hardware

- AMD Instinct GPU(s), such as MI300X.
- Enough aggregate HBM to serve `Qwen/Qwen3.6-35B-A3B` with tensor parallelism.
- Enough CPU DRAM for LMCache offload. Start with 64 GB if available, and increase for larger stress runs.

### Software

- ROCm installed and visible in containers.
- Docker configured for non-root usage.
- Hugging Face access for `Qwen/Qwen3.6-35B-A3B`.
- vLLM `>=0.19.0` is recommended by the Qwen model card.
- LMCache built from source with `BUILD_WITH_HIP=1` on AMD GPUs.

### Important AMD/LMCache notes

- Set `PYTHONHASHSEED=0` for cache-key consistency across vLLM workers and processes.
- Use `--enable-prefix-caching` with LMCache unless a specific experiment intentionally disables vLLM prefix caching.
- Do not enable `LMCACHE_SAVE_DECODE_CACHE=true` for these benchmarks; decode-cache offload can serialize the decode path and distort results.
- Use `--language-model-only` for text-only benchmarks to skip the vision encoder and leave more HBM for KV cache.

## Configuration

Adjust these variables for your server. The defaults target an 8-GPU MI300X-style node and keep the context window at Qwen3.6's native 262K tokens. For smaller systems, reduce `TENSOR_PARALLEL_SIZE`, `MAX_MODEL_LEN`, or `GPU_MEMORY_UTILIZATION`.

If you run this notebook from a host-local Jupyter server rather than inside the Docker container, the repo path might not be `/workspace/LMCache`. The configuration cell below detects the local checkout automatically. If you intentionally want to use `/workspace`, create it with sudo first:

```bash
sudo mkdir -p /workspace
sudo chown -R "$USER:$USER" /workspace
```

The notebook also falls back to a writable directory under your home folder if the selected results directory cannot be created.

In [2]:
import os
from pathlib import Path

MODEL_ID = "Qwen/Qwen3.6-35B-A3B"
TENSOR_PARALLEL_SIZE = 4
MAX_MODEL_LEN = 32768
GPU_MEMORY_UTILIZATION = 0.6
LMCACHE_CPU_GB = 128
PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# Detect the LMCache checkout whether Jupyter starts from the repo root,
# benchmarks/, or a mounted /workspace/LMCache directory.
def find_lmcache_repo() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent, Path("/workspace/LMCache")]
    for candidate in candidates:
        if (candidate / "benchmarks" / "long_doc_qa" / "long_doc_qa.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find LMCache repo. Start Jupyter from the LMCache checkout "
        "or mount it at /workspace/LMCache."
    )

LMCACHE_REPO = find_lmcache_repo()
RESULTS_DIR = LMCACHE_REPO / "benchmarks" / "qwen36_lmcache_results"

try:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
except PermissionError:
    # Host-local Jupyter sessions might not be allowed to create /workspace.
    RESULTS_DIR = Path.home() / "qwen36_lmcache_results"
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Permission denied for repo results directory. Using {RESULTS_DIR} instead.")

# Export paths so notebook shell cells can reuse the detected locations.
os.environ["LMCACHE_REPO"] = str(LMCACHE_REPO)
os.environ["QWEN36_RESULTS_DIR"] = str(RESULTS_DIR)
os.environ["MODEL_ID"] = MODEL_ID
os.environ["BASE_URL"] = BASE_URL

print(f"Model: {MODEL_ID}")
print(f"Tensor parallel size: {TENSOR_PARALLEL_SIZE}")
print(f"Max model length: {MAX_MODEL_LEN:,}")
print(f"Base URL: {BASE_URL}")
print(f"LMCache repository: {LMCACHE_REPO}")
print(f"Results directory: {RESULTS_DIR}")

Model: Qwen/Qwen3.6-35B-A3B
Tensor parallel size: 4
Max model length: 32,768
Base URL: http://localhost:8000/v1
LMCache repository: /root/LMcache/LMCache-AMD
Results directory: /root/LMcache/LMCache-AMD/benchmarks/qwen36_lmcache_results


## Step 1: Verify ROCm and Docker

Run these checks on the host before launching the container. On ROCm 6.4 and earlier, use `rocm-smi` if `amd-smi` is unavailable.

In [3]:
%%bash
set -e
if command -v amd-smi >/dev/null 2>&1; then
  amd-smi
elif command -v rocm-smi >/dev/null 2>&1; then
  rocm-smi
else
  echo "Neither amd-smi nor rocm-smi was found. Install ROCm tools or run inside a ROCm-enabled environment."
fi

# Uncomment this if you want to verify Docker access from this notebook kernel.
 docker run --rm hello-world

+------------------------------------------------------------------------------+
| AMD-SMI 26.2.2+97f5574fe2    amdgpu version: 6.16.13  ROCm version: 7.2.4    |
| VBIOS version: 00159017                                                      |
| Platform: Linux Guest                                                        |
|-------------------------------------+----------------------------------------|
| BDF                        GPU-Name | Mem-Uti   Temp   UEC       Power-Usage |
| GPU  HIP-ID  OAM-ID  Partition-Mode | GFX-Uti    Fan               Mem-Usage |
|=====================================+========================================|
| 0000:83:00.0 AMD Instinct MI300X VF | 0 %      51 °C   0           149/750 W |
|   0       0       1        SPX/NPS1 | 0 %        N/A           285/196288 MB |
|-------------------------------------+----------------------------------------|
| 0000:8b:00.0 AMD Instinct MI300X VF | 0 %      45 °C   0           146/750 W |
|   1       1       0       

## Step 2: Launch a ROCm vLLM container

The notebook runs on the host Jupyter server, so the ROCm/vLLM workload runs inside a Docker container named `LMcache`. The cells below mount the detected LMCache checkout and a Hugging Face cache directory into the container.

If a previous `LMcache` container exists, the launch cell removes it first so stale placeholder mounts do not break later steps.

In [4]:
%%bash  
docker pull rocm/vllm-dev:nightly



nightly: Pulling from rocm/vllm-dev
4f4fb700ef54: Pulling fs layer
4f4fb700ef54: Pulling fs layer
0d0da7500cbc: Pulling fs layer
4e8fd66612e7: Pulling fs layer
d11e69ce68b1: Pulling fs layer
3d70a770d19d: Pulling fs layer
275dd310cae3: Pulling fs layer
fb41b8cf3661: Pulling fs layer
c231352eb1d7: Pulling fs layer
6ecd160fd12b: Pulling fs layer
4f4fb700ef54: Pulling fs layer
b7a02185a4d4: Pulling fs layer
644cd9aedceb: Pulling fs layer
4f4fb700ef54: Pulling fs layer
4f4fb700ef54: Pulling fs layer
272972df534c: Pulling fs layer
b9ca9053fdb4: Pulling fs layer
e141e1b06a76: Pulling fs layer
4f4fb700ef54: Already exists
b7a02185a4d4: Download complete
3d70a770d19d: Download complete
c231352eb1d7: Download complete
b9ca9053fdb4: Download complete
4e8fd66612e7: Download complete
6ecd160fd12b: Download complete
25ffad080a1d: Download complete
644cd9aedceb: Download complete
0d0da7500cbc: Download complete
fb41b8cf3661: Download complete
275dd310cae3: Download complete
272972df534c: Download co

In [5]:
%%bash
set -e
# Launch the ROCm/vLLM container in detached mode. Do not use -it in notebook cells.
# Run as the host UID/GID so Git accepts the mounted checkout ownership.
REPO_DIR="${LMCACHE_REPO:-/home/haichzha@amd.com/lmcache/LMCache}"
HF_CACHE="${HF_HOME:-$HOME/.cache/huggingface}"
mkdir -p "$HF_CACHE"

docker rm -f LMcache >/dev/null 2>&1 || true

docker run -d \
  --ipc=host \
  --network=host \
  --privileged \
  --cap-add=CAP_SYS_ADMIN \
  --cap-add=SYS_PTRACE \
  --device=/dev/kfd \
  --device=/dev/dri \
  --device=/dev/mem \
  --group-add=video \
  --group-add=render \
  --security-opt seccomp=unconfined \
  --user "$(id -u):$(id -g)" \
  --env HOME=/workspace/hf-cache \
  --env USER=lmcache \
  --env LOGNAME=lmcache \
  --env TORCHINDUCTOR_CACHE_DIR=/workspace/hf-cache/torchinductor \
  --env HUGGINGFACE_HUB_CACHE=/workspace/hf-cache \
  --env HF_HOME=/workspace/hf-cache \
  --env SHELL=/bin/bash \
  -v "$REPO_DIR:/workspace/LMCache" \
  -v "$HF_CACHE:/workspace/hf-cache" \
  -w /workspace/LMCache \
  --name LMcache \
  rocm/vllm-dev:nightly \
  sleep infinity

docker ps --filter "name=LMcache"

cba7799924b208e7c44bca3d46f9e10a2c1ff2e9b18c903ebb1baf1d7f1c0f99
CONTAINER ID   IMAGE                   COMMAND            CREATED        STATUS                  PORTS     NAMES
cba7799924b2   rocm/vllm-dev:nightly   "sleep infinity"   1 second ago   Up Less than a second             LMcache


In [6]:
%%bash
set -e

# Jupyter bash cells do not allocate an interactive TTY, so do not use:
#   docker exec -it LMcache /bin/bash
# Instead, run non-interactive commands with docker exec.

docker ps --filter "name=LMcache"
docker exec LMcache bash -lc 'cd /workspace/LMCache && pwd && python --version'

# To open an interactive shell, run this from a real terminal, not from a notebook cell:
#   docker exec -it LMcache /bin/bash

CONTAINER ID   IMAGE                   COMMAND            CREATED         STATUS         PORTS     NAMES
cba7799924b2   rocm/vllm-dev:nightly   "sleep infinity"   6 seconds ago   Up 5 seconds             LMcache
/workspace/LMCache
Python 3.12.13


## Step 3: Build LMCache for AMD GPUs

Inside the container, build LMCache from source with HIP enabled. This avoids CUDA-linked wheels on ROCm systems and ensures the LMCache native operators load correctly.

In [7]:
%%bash
set -e
# Build and install inside the ROCm/vLLM container. The container runs as the
# host UID/GID, so install Python packages into the user site-packages.
docker exec LMcache bash -lc '
  set -e
  cd /workspace/LMCache
  python -m pip install --user -U pip
  python -m pip uninstall -y nixl nixl-cu12 cupy-cuda12x cufile-python cuda-pathfinder || true
  SETUPTOOLS_SCM_PRETEND_VERSION_FOR_LMCACHE=0.0.0+ci BUILD_WITH_HIP=1 python -m pip install --user -e . --no-build-isolation
  python -m pip install --user -r benchmarks/multi_round_qa/requirements.txt
  python -m pip install --user openai pandas matplotlib huggingface_hub
'

Found existing installation: cufile-python 0.2.0
Uninstalling cufile-python-0.2.0:
  Successfully uninstalled cufile-python-0.2.0


Obtaining file:///workspace/LMCache
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Using cached cufile_python-0.2.0-py3-none-any.whl.metadata (1.5 kB)
Using cached cufile_python-0.2.0-py3-none-any.whl (5.3 kB)
  Building editable for lmcache (pyproject.toml): started
  Building editable for lmcache (pyproject.toml): still running...
  Building editable for lmcache (pyproject.toml): finished with status 'done'
  Created wheel for lmcache: filename=lmcache-0.0.0+ci-0.editable-cp312-cp312-linux_x86_64.whl size=12127 sha256=81126bd81b3674981893096cdea8fdb04ec147057709ada18b5ad16a0b21d766
  Stored in directory: /tmp/pip-ephem-wheel-cache-k1s4p0_j/wheels/2c/5c/58/c0685ffc800d246a2f438fbacab5e32b535db8a5f7be0149cf
Successfully built lmcache
  Attempting uninstall: lmc

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


## Step 4: Log in to Hugging Face

Qwen3.6-35B-A3B is served directly from Hugging Face in this tutorial. If your environment already has `HF_TOKEN` configured, this cell can be skipped.

In [ ]:
import os
from huggingface_hub import HfApi

# Non-interactive token check. Set HF_TOKEN in the environment before running
# private or gated model downloads. Public metadata checks can still work without it.
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
try:
    if hf_token:
        user_info = HfApi(token=hf_token).whoami()
        print(f"Token validated successfully. Logged in as: {user_info['name']}")
    else:
        info = HfApi().model_info(MODEL_ID)
        print(f"No HF_TOKEN set. Public model metadata is reachable for: {info.modelId}")
except Exception as exc:
    print(f"Hugging Face check failed: {exc}")
    print("If the model download is gated in your environment, export HF_TOKEN before serving.")

## Step 5: Choose a serving configuration

Run **one** of the following server commands in a separate terminal inside the same container. Stop the running server with `Ctrl+C` before switching to another configuration.

The three configurations are:

- **A. No prefix cache**: lower-bound baseline. Every prefill is recomputed.
- **B. vLLM HBM prefix cache**: uses vLLM's GPU-resident prefix cache.
- **C. LMCache CPU DRAM tier**: uses vLLM prefix hashing plus LMCache KV offload/reuse through `LMCacheConnectorV1`.

**Workflow: Click each of the following experiment: A, B, and C, sequentially. After selecting an experiment, proceed to Step 6: wait for the server to start, send a smoke test, and run the benchmark test.
Then continue with either Step 7: Long Document QA Benchmark or Step 8: Multi-round QA Benchmark.
After Experiment A is completed, return to Experiment B, and then Experiment C.**

A. No prefix cache: lower-bound baseline. Every prefill is recomputed.

In [8]:
%%bash
set -e
# A. Baseline: no vLLM prefix cache and no LMCache.
docker exec -u 0 LMcache bash -lc 'pids=$(pgrep -f "[v]llm serve" || true); if [ -n "$pids" ]; then echo "Stopping existing vLLM server PIDs: $pids"; kill $pids; sleep 5; fi'
docker exec -u 0 -d LMcache bash -lc '
  cd /workspace/LMCache
  export PYTHONPATH=/workspace/hf-cache/.local/lib/python3.12/site-packages:$PYTHONPATH
  PYTHONHASHSEED=0 \
  VLLM_ROCM_USE_AITER=1 \
  SAFETENSORS_FAST_GPU=1 \
  vllm serve Qwen/Qwen3.6-35B-A3B \
    --port 8000 \
    --tensor-parallel-size 4 \
    --max-model-len 32768 \
    --gpu-memory-utilization 0.7 \
    --no-enable-prefix-caching \
    --trust-remote-code \
    > /workspace/LMCache/benchmarks/qwen36_lmcache_results/vllm_no_cache.log 2>&1
'
echo "Started baseline vLLM server in the LMcache container."

Started baseline vLLM server in the LMcache container.


B. vLLM HBM prefix cache: uses vLLM's GPU-resident prefix cache.

In [ ]:
%%bash
set -e
# B. vLLM HBM prefix cache only.
docker exec -u 0 LMcache bash -lc 'pids=$(pgrep -f "[v]llm serve" || true); if [ -n "$pids" ]; then echo "Stopping existing vLLM server PIDs: $pids"; kill $pids; sleep 5; fi'
docker exec -u 0 -d LMcache bash -lc '
  cd /workspace/LMCache
  export PYTHONPATH=/workspace/hf-cache/.local/lib/python3.12/site-packages:$PYTHONPATH
  PYTHONHASHSEED=0 \
  VLLM_ROCM_USE_AITER=1 \
  SAFETENSORS_FAST_GPU=1 \
  vllm serve Qwen/Qwen3.6-35B-A3B \
    --port 8000 \
    --tensor-parallel-size 4 \
    --max-model-len 32768 \
    --gpu-memory-utilization 0.7 \
    --enable-prefix-caching \
    --trust-remote-code \
    > /workspace/LMCache/benchmarks/qwen36_lmcache_results/vllm_hbm_prefix.log 2>&1
'
echo "Started vLLM HBM prefix-cache server in the LMcache container."

C. LMCache CPU DRAM tier: uses vLLM prefix hashing plus LMCache KV offload/reuse through LMCacheConnectorV1.

In [ ]:
%%bash
set -e
# C. LMCache CPU DRAM tier. This is the main LMCache configuration.
docker exec -u 0 LMcache bash -lc 'pids=$(pgrep -f "[v]llm serve" || true); if [ -n "$pids" ]; then echo "Stopping existing vLLM server PIDs: $pids"; kill $pids; sleep 5; fi'
docker exec -u 0 -d LMcache bash -lc '
  cd /workspace/LMCache
  export PYTHONPATH=/workspace/hf-cache/.local/lib/python3.12/site-packages:$PYTHONPATH
  PYTHONHASHSEED=0 \
  VLLM_ROCM_USE_AITER=1 \
  SAFETENSORS_FAST_GPU=1 \
  LMCACHE_LOCAL_CPU=true \
  LMCACHE_CHUNK_SIZE=256 \
  LMCACHE_MAX_LOCAL_CPU_SIZE=64 \
  vllm serve Qwen/Qwen3.6-35B-A3B \
    --port 8000 \
    --tensor-parallel-size 4 \
    --max-model-len 32768 \
    --gpu-memory-utilization 0.7 \
    --trust-remote-code \
    --enable-prefix-caching \
    --kv-transfer-config "{\"kv_connector\":\"LMCacheConnectorV1\",\"kv_role\":\"kv_both\"}" \
    > /workspace/LMCache/benchmarks/qwen36_lmcache_results/vllm_lmcache.log 2>&1
'
echo "Started LMCache-backed vLLM server in the LMcache container."

## Step 6: Wait for the server and send a smoke test

The server is ready when `GET /v1/models` succeeds. The smoke test sends a small OpenAI-compatible chat request to confirm that the endpoint, model name, and tokenizer are working.

In [9]:
import requests
import time

models_url = f"{BASE_URL}/models"
for attempt in range(60):
    try:
        response = requests.get(models_url, timeout=5)
        if response.ok:
            print("Server is ready:")
            print(response.json())
            break
    except requests.RequestException:
        pass
    print(f"Waiting for server... attempt {attempt + 1}/60")
    time.sleep(10)
else:
    raise RuntimeError(f"Server did not become ready at {models_url}")

Waiting for server... attempt 1/60
Waiting for server... attempt 2/60
Waiting for server... attempt 3/60
Waiting for server... attempt 4/60
Waiting for server... attempt 5/60
Waiting for server... attempt 6/60
Waiting for server... attempt 7/60
Waiting for server... attempt 8/60
Waiting for server... attempt 9/60
Waiting for server... attempt 10/60
Waiting for server... attempt 11/60
Waiting for server... attempt 12/60
Waiting for server... attempt 13/60
Waiting for server... attempt 14/60
Waiting for server... attempt 15/60
Waiting for server... attempt 16/60
Waiting for server... attempt 17/60
Waiting for server... attempt 18/60
Waiting for server... attempt 19/60
Waiting for server... attempt 20/60
Waiting for server... attempt 21/60
Waiting for server... attempt 22/60
Waiting for server... attempt 23/60
Waiting for server... attempt 24/60
Server is ready:
{'object': 'list', 'data': [{'id': 'Qwen/Qwen3.6-35B-A3B', 'object': 'model', 'created': 1782271363, 'owned_by': 'vllm', 'root':

In [12]:
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key="EMPTY")

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": "You are a concise AI infrastructure assistant."},
        {"role": "user", "content": "In one sentence, explain why KV cache reuse reduces TTFT."},
    ],
    temperature=0.0,
    max_tokens=96,
)

print(response.choices[0].message.content)

Here's a thinking process:

1.  **Analyze User Input:**
   - **Topic:** KV cache reuse
   - **Metric:** TTFT (Time To First Token)
   - **Question:** Why does KV cache reuse reduce TTFT?
   - **Constraint:** One sentence

2.  **Define Key Concepts:**
   - **KV Cache:** Stores key and value tensors from previous attention computations in LLMs to avoid recomputing


## Step 7: Long Document QA benchmark

This benchmark sends synthetic long-document prompts through the OpenAI-compatible endpoint. Each document is warmed once, then repeated according to the selected repeat mode. With LMCache enabled, the second round should reuse KV cache for repeated document prefixes and reduce TTFT.

The blog used 100 documents, 10,000-token documents, 300 output tokens, and up to four in-flight requests. Start with a smaller smoke setting, then scale up to the blog-style run.

In [15]:
import sys, os
os.environ["PYTHON_BIN"] = sys.executable

In [18]:
%%bash
set -e
# Long Document QA smoke test. Use this first to validate the benchmark path.
cd "${LMCACHE_REPO:-/workspace/LMCache}"
mkdir -p "${QWEN36_RESULTS_DIR:-benchmarks/qwen36_lmcache_results}"
"${PYTHON_BIN:-python3}" benchmarks/long_doc_qa/long_doc_qa.py \
  --model "${MODEL_ID:-Qwen/Qwen3.6-35B-A3B}" \
  --base-url "${BASE_URL:-http://localhost:8000/v1}" \
  --num-documents 6 \
  --document-length 8192 \
  --output-len 64 \
  --repeat-count 1 \
  --repeat-mode tile \
  --max-inflight-requests 2 \
  --output "${QWEN36_RESULTS_DIR:-benchmarks/qwen36_lmcache_results}/long_doc_smoke.txt"

Using base URL: http://localhost:8000/v1
Warmup round mean TTFT: 0.259s
Warmup round time: 3.488s
Warmup round prompt count: 6
Warmup round successful prompt count: 6

=== BENCHMARK RESULTS ===
Query round mean TTFT: 0.257s
Query round time: 3.476s
Query round prompt count: 6
Query round successful prompt count: 6


please modify the result files names for each round of benchmark test.                         Recommended output names: long_doc_no_cache.txt, long_doc_hbm_prefix.txt, long_doc_lmcache.txt.
Examples: --output "${QWEN36_RESULTS_DIR:-benchmarks/qwen36_lmcache_results}/long_doc_no_lmcache.txt"

In [20]:
%%bash
set -e
# Blog-style Long Document QA run. Run this separately for each server configuration.
# Recommended output names: long_doc_no_cache.txt, long_doc_hbm_prefix.txt, long_doc_lmcache.txt
cd "${LMCACHE_REPO:-/workspace/LMCache}"
mkdir -p "${QWEN36_RESULTS_DIR:-benchmarks/qwen36_lmcache_results}"
"${PYTHON_BIN:-python3}" benchmarks/long_doc_qa/long_doc_qa.py \
  --model "${MODEL_ID:-Qwen/Qwen3.6-35B-A3B}" \
  --base-url "${BASE_URL:-http://localhost:8000/v1}" \
  --num-documents 100 \
  --document-length 10000 \
  --output-len 300 \
  --repeat-count 1 \
  --repeat-mode tile \
  --max-inflight-requests 8 \
  --output "${QWEN36_RESULTS_DIR:-benchmarks/qwen36_lmcache_results}/long_doc_no_lmcache.txt"

Using base URL: http://localhost:8000/v1
Warmup round mean TTFT: 0.772s
Warmup round time: 86.877s
Warmup round prompt count: 100
Warmup round successful prompt count: 100

=== BENCHMARK RESULTS ===
Query round mean TTFT: 0.771s
Query round time: 86.796s
Query round prompt count: 100
Query round successful prompt count: 100


### Check LMCache hits

When running the LMCache configuration, watch the vLLM server logs for lines similar to:

```text
LMCache hit tokens: <N>, need to load: <M>
Retrieved <N> out of total <T> tokens
```

A repeated prompt showing `LMCache hit tokens: 0` usually means one of the cache-key requirements is wrong. Check `PYTHONHASHSEED=0`, `--enable-prefix-caching`, model name consistency, and whether the second request actually shares a prefix with the first.

## Step 8: Multi-round QA benchmark

This benchmark simulates multiple users carrying stateful conversations. Each user's later turns include prior conversation history, so long shared prefixes and user-specific history can be reused or evicted depending on the cache strategy.

The blog used 20 users and 6 rounds. The smoke test below is smaller; use the blog-style run after validating your server.

In [21]:
%%bash
set -e
# Multi-round QA smoke test.
cd "${LMCACHE_REPO:-/workspace/LMCache}/benchmarks/multi_round_qa"
mkdir -p "${QWEN36_RESULTS_DIR:-../qwen36_lmcache_results}"
"${PYTHON_BIN:-python3}" multi-round-qa.py \
  --num-users 3 \
  --num-rounds 2 \
  --qps 0.5 \
  --shared-system-prompt 1000 \
  --user-history-prompt 2000 \
  --answer-len 64 \
  --model "${MODEL_ID:-Qwen/Qwen3.6-35B-A3B}" \
  --base-url "${BASE_URL:-http://localhost:8000/v1}" \
  --time 180 \
  --output "${QWEN36_RESULTS_DIR:-../qwen36_lmcache_results}/multi_round_smoke.csv"

[2026-06-24 06:53:28,126] DEBUG: Starting the asyncio loop (utils.py:94:AsyncLoopWrapper)
[2026-06-24 06:53:28,126] INFO: Warming up the engine (multi-round-qa.py:567:__main__)
[2026-06-24 06:53:28,320] INFO: Waiting for 10 tasks to finish (utils.py:71:AsyncLoopWrapper)
[2026-06-24 06:53:29,201] INFO: Gap between users: 4.0 secs.
Gap between user reqs: 8.0 secs.
Expected length of user session: 16.0 secs. (multi-round-qa.py:369:__main__)
[2026-06-24 06:53:29,201] INFO: Joined a new user 5, now active users: 5 (multi-round-qa.py:459:__main__)
[2026-06-24 06:53:29,201] INFO: Removing 1 finished sessions, now active users: 4 (multi-round-qa.py:427:__main__)
[2026-06-24 06:53:33,206] INFO: Joined a new user 6, now active users: 5 (multi-round-qa.py:459:__main__)
[2026-06-24 06:53:34,007] INFO: Removing 1 finished sessions, now active users: 4 (multi-round-qa.py:427:__main__)
[2026-06-24 06:53:37,212] INFO: Joined a new user 7, now active users: 5 (multi-round-qa.py:459:__main__)
[2026-06-2



==================== Performance summary ======================
  Config QPS: 0.5000 reqs/s

  Actual QPS: 0.7323 reqs/s

  Processing speed: 0.7323 reqs/s

  Requests on-the-fly: 0

  Input tokens per second: 2278.5698 tokens/s

  Output tokens per second: 46.8662 tokens/s

  Average generation throughput (per request): 116.8469 tokens/req/s

  Average TTFT: 0.1803s

Time range: 1782284009.2013693 - 1782284039.244338 (30.04s)




==================== Performance summary ======================
  Config QPS: 0.5000 reqs/s

  Actual QPS: 0.6990 reqs/s

  Processing speed: 0.6990 reqs/s

  Requests on-the-fly: 0

  Input tokens per second: 2196.1930 tokens/s

  Output tokens per second: 44.7354 tokens/s

  Average generation throughput (per request): 115.3389 tokens/req/s

  Average TTFT: 0.1826s

Time range: 1782284039.2525394 - 1782284069.2958913 (30.04s)




==================== Performance summary ======================
  Config QPS: 0.5000 reqs/s

  Actual QPS: 0.7989 reqs/s

  Pro

In [ ]:
%%bash
set -e
# Blog-style Multi-round QA run. Run this separately for each server configuration.
# Recommended output names: multi_round_no_cache.csv, multi_round_hbm_prefix.csv, multi_round_lmcache.csv
cd "${LMCACHE_REPO:-/workspace/LMCache}/benchmarks/multi_round_qa"
mkdir -p "${QWEN36_RESULTS_DIR:-../qwen36_lmcache_results}"
"${PYTHON_BIN:-python3}" multi-round-qa.py \
  --num-users 6 \
  --num-rounds 3 \
  --qps 1 \
  --shared-system-prompt 1000 \
  --user-history-prompt 2000 \
  --answer-len 100 \
  --time 180 \
  --model "${MODEL_ID:-Qwen/Qwen3.6-35B-A3B}" \
  --base-url "${BASE_URL:-http://localhost:8000/v1}" \
  --output "${QWEN36_RESULTS_DIR:-../qwen36_lmcache_results}/multi_round_lmcache.csv"

## Step 9: Summarize Multi-round QA results

The multi-round QA script writes per-request rows to CSV. Use the following helper to compare output files from different server configurations.

In [ ]:
import os
import pandas as pd
from pathlib import Path

result_dir = Path(os.environ.get("QWEN36_RESULTS_DIR", str(RESULTS_DIR))).expanduser()
files = {
    "no_cache": result_dir / "multi_round_no_cache.csv",
    "hbm_prefix": result_dir / "multi_round_hbm_prefix.csv",
    "lmcache": result_dir / "multi_round_lmcache.csv",
}

summary_rows = []
for name, path in files.items():
    if not path.exists():
        print(f"Skipping {name}: {path} not found")
        continue
    df = pd.read_csv(path)
    numeric_cols = {col.lower(): col for col in df.columns}
    ttft_col = numeric_cols.get("ttft") or numeric_cols.get("ttfts") or numeric_cols.get("time_to_first_token")
    prompt_col = numeric_cols.get("prompt_tokens") or numeric_cols.get("prompt_len")
    gen_col = numeric_cols.get("generation_tokens") or numeric_cols.get("output_tokens")
    start_col = numeric_cols.get("launch_time") or numeric_cols.get("start_time")
    finish_col = numeric_cols.get("finish_time") or numeric_cols.get("end_time")

    row = {"config": name, "requests": len(df)}
    if ttft_col:
        row["ttft_avg_s"] = df[ttft_col].mean()
        row["ttft_p50_s"] = df[ttft_col].quantile(0.50)
        row["ttft_p95_s"] = df[ttft_col].quantile(0.95)
        row["ttft_max_s"] = df[ttft_col].max()
    if prompt_col and start_col and finish_col:
        duration = max(df[finish_col].max() - df[start_col].min(), 1e-9)
        row["input_tok_s"] = df[prompt_col].sum() / duration
    if gen_col and start_col and finish_col:
        duration = max(df[finish_col].max() - df[start_col].min(), 1e-9)
        row["output_tok_s"] = df[gen_col].sum() / duration
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary

In [ ]:
import matplotlib.pyplot as plt

if "summary" in globals() and not summary.empty:
    ax = summary.set_index("config")[[col for col in ["ttft_avg_s", "ttft_p95_s"] if col in summary.columns]].plot(
        kind="bar",
        figsize=(8, 4),
        title="Multi-round QA TTFT comparison",
    )
    ax.set_ylabel("seconds")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
else:
    print("No summary data available yet. Run the benchmark commands first.")

## Step 10: Interpret the results

Use these questions to interpret your runs:

- **Does the working set fit in HBM?** If yes, vLLM HBM prefix cache can match or beat LMCache because no CPU transfer is needed.
- **Does LMCache show nonzero hit tokens on repeated prompts?** If not, fix cache-key consistency before trusting benchmark numbers.
- **Does TTFT improve more than end-to-end latency?** That is expected. KV reuse primarily reduces prefill work; decode speed can still dominate total latency.
- **Does LMCache improve p95 or max TTFT under stress?** This is the key long-context production signal.
- **Are prompt serialization and model names byte-stable across rounds?** Small changes to prompt text or model identifier can prevent cache reuse.

For Qwen3.6-35B-A3B, start with the native 262K context only if your node has enough HBM. For smaller systems, reduce `--max-model-len` while preserving the same comparison methodology.

## Troubleshooting

### The server never becomes ready

- Confirm ROCm devices are visible with `amd-smi` or `rocm-smi`.
- Reduce `--max-model-len`, `--gpu-memory-utilization`, or tensor-parallel size if vLLM reports OOM.
- Confirm the container has access to Hugging Face credentials.

### LMCache reports zero hit tokens on repeated prompts

- Ensure every process uses `PYTHONHASHSEED=0`.
- Use `--enable-prefix-caching` with `LMCacheConnectorV1`.
- Confirm all requests use the exact same model string.
- Confirm the repeated requests share byte-identical prompt prefixes.

### LMCache is slower than vLLM prefix cache

This can happen when the active KV working set fits in GPU HBM. Increase document length, number of users, number of documents, or benchmark duration to create HBM pressure. LMCache is most useful when CPU DRAM prevents useful KV blocks from being evicted from the overall serving system.

## Next steps

- Increase `LMCACHE_MAX_LOCAL_CPU_SIZE` and rerun the stress benchmarks.
- Try `--max-model-len 131072` as a lower-memory variant and compare the crossover point.
- Add FP8 KV cache settings if your vLLM/Qwen3.6 stack supports them on your ROCm release.
- Use LMCache observability examples to monitor cache hit rate, read/write throughput, and request traces.

## Step 11: KV Cache Calculator GUI

Use this calculator to estimate KV cache memory before choosing `MAX_MODEL_LEN`, `TENSOR_PARALLEL_SIZE`, and `LMCACHE_MAX_LOCAL_CPU_SIZE`.

The default preset is for `Qwen/Qwen3.6-35B-A3B` and estimates the attention KV cache using the model card values: 10 effective gated-attention KV layers, 2 KV heads, and 256 head dimension. This is an approximation for planning; the exact runtime allocation can differ because Qwen3.6 is a hybrid architecture and vLLM may allocate additional state for non-standard layers.

The formulas follow the same idea as the LMCache KV Cache Size Calculator:

```text
KV elements = 2 × KV layers × tokens × KV heads × head_dim
KV GB = KV elements × dtype_bytes / 1024³
Per-GPU/rank KV GB ≈ KV GB / tensor_parallel_size
```

In [22]:
import math

import ipywidgets as widgets
from IPython.display import HTML, display

DTYPE_BYTES = {
    "float32": 4,
    "float16": 2,
    "bfloat16": 2,
    "fp8/int8": 1,
}

MODEL_PRESETS = {
    "Qwen/Qwen3.6-35B-A3B (attention KV estimate)": {
        "kv_layers": 10,
        "kv_heads": 2,
        "head_dim": 256,
        "note": (
            "Qwen3.6 hybrid-layout estimate: 10 gated-attention blocks, "
            "2 KV heads, head_dim 256. Runtime allocation can differ."
        ),
    },
    "Qwen/Qwen3-32B": {
        "kv_layers": 64,
        "kv_heads": 8,
        "head_dim": 128,
        "note": "Uniform-attention Qwen3 preset from LMCache calculator modelconfig.",
    },
    "Llama-3.1-8B-Instruct": {
        "kv_layers": 32,
        "kv_heads": 8,
        "head_dim": 128,
        "note": "GQA Llama-style reference preset.",
    },
    "Custom": {
        "kv_layers": 40,
        "kv_heads": 8,
        "head_dim": 128,
        "note": "Enter model-specific KV layer/head dimensions manually.",
    },
}

preset = widgets.Dropdown(
    options=list(MODEL_PRESETS),
    value="Qwen/Qwen3.6-35B-A3B (attention KV estimate)",
    description="Model",
    layout=widgets.Layout(width="620px"),
)

tokens = widgets.IntText(
    value=32_768,
    description="Tokens",
    layout=widgets.Layout(width="260px"),
)
kv_layers = widgets.IntText(
    value=MODEL_PRESETS[preset.value]["kv_layers"],
    description="KV layers",
    layout=widgets.Layout(width="260px"),
)
kv_heads = widgets.IntText(
    value=MODEL_PRESETS[preset.value]["kv_heads"],
    description="KV heads",
    layout=widgets.Layout(width="260px"),
)
head_dim = widgets.IntText(
    value=MODEL_PRESETS[preset.value]["head_dim"],
    description="Head dim",
    layout=widgets.Layout(width="260px"),
)
dtype = widgets.Dropdown(
    options=list(DTYPE_BYTES),
    value="bfloat16",
    description="Dtype",
    layout=widgets.Layout(width="260px"),
)
tp_size = widgets.IntSlider(
    value=4,
    min=1,
    max=16,
    step=1,
    description="TP size",
    layout=widgets.Layout(width="420px"),
)
mem_budget_per_gpu = widgets.FloatText(
    value=64.0,
    description="GB/GPU",
    layout=widgets.Layout(width="260px"),
)
cpu_budget_per_rank = widgets.FloatText(
    value=64.0,
    description="CPU GB/rank",
    layout=widgets.Layout(width="260px"),
)
output = widgets.Output()


def _positive(value, default=1):
    try:
        return max(float(value), default)
    except Exception:
        return default


def kv_cache_gb(num_tokens, num_layers, num_kv_heads, dim, bytes_per_elem):
    total_elements = 2 * num_layers * num_tokens * num_kv_heads * dim
    total_bytes = total_elements * bytes_per_elem
    return total_bytes / (1024**3), total_elements, total_bytes


def update_from_preset(change=None):
    cfg = MODEL_PRESETS[preset.value]
    kv_layers.value = cfg["kv_layers"]
    kv_heads.value = cfg["kv_heads"]
    head_dim.value = cfg["head_dim"]
    calculate()


def calculate(change=None):
    with output:
        output.clear_output()
        n_tokens = int(_positive(tokens.value))
        layers = int(_positive(kv_layers.value))
        heads = int(_positive(kv_heads.value))
        dim = int(_positive(head_dim.value))
        bytes_per_elem = DTYPE_BYTES[dtype.value]
        tp = int(_positive(tp_size.value))
        gpu_budget = _positive(mem_budget_per_gpu.value, 0.0)
        cpu_budget = _positive(cpu_budget_per_rank.value, 0.0)

        total_gb, total_elements, total_bytes = kv_cache_gb(
            n_tokens, layers, heads, dim, bytes_per_elem
        )
        per_gpu_gb = total_gb / tp
        gb_per_1k_total = total_gb / max(n_tokens / 1000, 1e-9)
        gb_per_1k_per_gpu = per_gpu_gb / max(n_tokens / 1000, 1e-9)

        max_tokens_gpu = int(n_tokens * gpu_budget / per_gpu_gb) if per_gpu_gb > 0 else 0
        max_tokens_cpu_rank = int(n_tokens * cpu_budget / per_gpu_gb) if per_gpu_gb > 0 else 0
        recommended_lmcache_cpu = max(cpu_budget, per_gpu_gb)

        note = MODEL_PRESETS[preset.value]["note"]
        html = f"""
        <div style="border:1px solid #ddd; border-radius:8px; padding:14px; max-width:900px;">
          <h3 style="margin-top:0;">KV Cache Estimate</h3>
          <p><b>Preset note:</b> {note}</p>
          <table style="border-collapse:collapse; width:100%;">
            <tr><td><b>Formula</b></td><td>2 × {layers} × {n_tokens:,} × {heads} × {dim}</td></tr>
            <tr><td><b>Total elements</b></td><td>{total_elements:,}</td></tr>
            <tr><td><b>Dtype size</b></td><td>{bytes_per_elem} bytes</td></tr>
            <tr><td><b>Total KV cache</b></td><td>{total_gb:.4f} GB</td></tr>
            <tr><td><b>Per GPU/rank KV cache</b></td><td>{per_gpu_gb:.4f} GB with TP={tp}</td></tr>
            <tr><td><b>Total GB per 1K tokens</b></td><td>{gb_per_1k_total:.4f} GB</td></tr>
            <tr><td><b>Per-GPU GB per 1K tokens</b></td><td>{gb_per_1k_per_gpu:.4f} GB</td></tr>
            <tr><td><b>Max tokens for GPU budget</b></td><td>≈ {max_tokens_gpu:,} tokens at {gpu_budget:.2f} GB/GPU</td></tr>
            <tr><td><b>Max tokens for LMCache CPU budget</b></td><td>≈ {max_tokens_cpu_rank:,} tokens at {cpu_budget:.2f} CPU GB/rank</td></tr>
            <tr><td><b>Suggested LMCACHE_MAX_LOCAL_CPU_SIZE</b></td><td>Start around {recommended_lmcache_cpu:.1f} GB/rank, then tune with benchmark results.</td></tr>
          </table>
          <p style="margin-bottom:0;"><i>Assumption:</i> per-GPU/rank values divide total attention KV by tensor parallel size. Hybrid models may allocate additional runtime state.</p>
        </div>
        """
        display(HTML(html))


preset.observe(update_from_preset, names="value")
for widget in [tokens, kv_layers, kv_heads, head_dim, dtype, tp_size, mem_budget_per_gpu, cpu_budget_per_rank]:
    widget.observe(calculate, names="value")

controls = widgets.VBox(
    [
        preset,
        widgets.HBox([tokens, dtype]),
        widgets.HBox([kv_layers, kv_heads, head_dim]),
        widgets.HBox([tp_size]),
        widgets.HBox([mem_budget_per_gpu, cpu_budget_per_rank]),
    ]
)

display(widgets.VBox([controls, output]))
calculate()